In [1]:
# %pip install -q --upgrade pip
# %pip install -q --upgrade tensorflow tensorflow-datasets transformers accelerate evaluate
# Il est recommandé de redémarrer l'environnement d'exécution (Runtime -> Restart runtime) après cette installation.

In [2]:
# import transformers

# print(transformers.__version__)
# print(hasattr(transformers, "TFBertForSequenceClassification"))
# print(hasattr(transformers, "BertTokenizer"))

# %pip install -U transformers

import platform
import transformers
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

c:\Users\meles\Documents\TTA_DI_BootCamp_Gilles-Chris_MAKE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Python version      : 3.12.10
TensorFlow version  : 2.21.0
GPU devices detected: []


In [6]:
import tensorflow_datasets as tfds

print(tfds)
print("Version :", tfds.__version__)
print("load existe :", hasattr(tfds, "load"))
print(dir(tfds)[:20])

<module 'tensorflow_datasets' from 'c:\\Users\\meles\\Documents\\TTA_DI_BootCamp_Gilles-Chris_MAKE\\.venv\\Lib\\site-packages\\tensorflow_datasets\\__init__.py'>
Version : 4.9.10
load existe : True
['GenerateMode', 'ImageFolder', 'ReadConfig', 'Split', 'TranslateFolder', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'annotations', 'as_dataframe', 'as_numpy', 'audio']


In [ ]:
import tensorflow_datasets as tfds

(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)
print(ds_info)

Dl Completed...:   0%|          | 0/1 [00:50<?, ? url/s]

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

In [ ]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

In [ ]:
EPOCHS = 2  # increase to 3 if time allows
history = model.fit(
#To-Do
)